# Split estratificado e baseline

In [1]:
import os
import json
import time

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

CURRDIR = os.getcwd()
DATADIR = os.path.join(CURRDIR, "..", "data")
RESDIR = os.path.join(CURRDIR, "..", "resultados")
SUBDIR = os.path.join(CURRDIR, "..", "submissoes")
SPLITDIR = os.path.join(CURRDIR, "..", "splits")
SEED = 42

np.random.seed(SEED)

## Dados

In [2]:
treino_completo = pd.read_csv(os.path.join(DATADIR, "train.csv"))
teste = pd.read_csv(os.path.join(DATADIR, "test.csv"))
print(treino_completo.shape, teste.shape)
treino_completo.head()

(12120, 6) (5195, 5)


,id,premise,hypothesis,lang_abv,language,label
0,5130fd2cb5,and these comments were considered in formulat...,The rules developed in the interim were put to...,en,English,0
1,5b72532a0b,These are issues that we wrestle with in pract...,Practice groups are not permitted to work on t...,en,English,2
2,3931fbe82a,Des petites choses comme celles-là font une di...,J'essayais d'accomplir quelque chose.,fr,French,0
3,5622f0c60b,you know they can't really defend themselves l...,They can't defend themselves because of their ...,en,English,0
4,86aaa48b45,ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...,เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร,th,Thai,1


## Split

In [3]:
estrato = treino_completo["label"].astype(str) + "_" + treino_completo["lang_abv"]
contagem_estrato = estrato.value_counts()
print("estratos:", contagem_estrato.size)
print("menor estrato:", contagem_estrato.idxmin(), contagem_estrato.min())

estratos: 45
menor estrato: 0_de 108


In [4]:
ids_treino, ids_val = train_test_split(
    treino_completo["id"], test_size=0.2, random_state=SEED, stratify=estrato
)

holdout = pd.DataFrame({"id": treino_completo["id"]})
holdout["conjunto"] = np.where(holdout["id"].isin(ids_val), "validacao", "treino")
holdout.to_csv(os.path.join(SPLITDIR, "holdout.csv"), index=False)
holdout["conjunto"].value_counts()

conjunto
treino       9696
validacao    2424
Name: count, dtype: int64

In [5]:
df = treino_completo.merge(holdout, on="id")
treino = df[df["conjunto"] == "treino"]
val = df[df["conjunto"] == "validacao"]

prop_label = pd.DataFrame({
    "treino": treino["label"].value_counts(normalize=True),
    "validacao": val["label"].value_counts(normalize=True),
}).sort_index().round(4)
prop_label

,treino,validacao
label,,
0,0.3445,0.3449
1,0.3202,0.3197
2,0.3353,0.3354


In [6]:
prop_lang = pd.DataFrame({
    "treino": treino["lang_abv"].value_counts(normalize=True),
    "validacao": val["lang_abv"].value_counts(normalize=True),
}).sort_values("treino", ascending=False).round(4)
prop_lang["diferenca"] = (prop_lang["treino"] - prop_lang["validacao"]).abs()
prop_lang

,treino,validacao,diferenca
lang_abv,,,
en,0.5669,0.5664,0.0005
zh,0.0339,0.0338,0.0001
ar,0.0330,0.0334,0.0004
fr,0.0321,0.0326,0.0005
sw,0.0318,0.0318,0.0000
ur,0.0314,0.0318,0.0004
vi,0.0312,0.0314,0.0002
ru,0.0310,0.0309,0.0001
hi,0.0309,0.0305,0.0004


## Treino

In [7]:
colunas_x = ["premise", "hypothesis"]
modelo = DummyClassifier(strategy="most_frequent")

inicio = time.time()
modelo.fit(treino[colunas_x], treino["label"])
tempo_treino = time.time() - inicio
print("classe majoritaria:", modelo.classes_[np.argmax(modelo.class_prior_)])

classe majoritaria: 0


## Validação

In [8]:
pred_treino = modelo.predict(treino[colunas_x])
pred_val = modelo.predict(val[colunas_x])
prob_val = modelo.predict_proba(val[colunas_x])

acc_treino = accuracy_score(treino["label"], pred_treino)
acc_val = accuracy_score(val["label"], pred_val)
print(f"acuracia treino: {acc_treino:.4f}")
print(f"acuracia validacao: {acc_val:.4f}")
print(classification_report(val["label"], pred_val, digits=4, zero_division=0))

acuracia treino: 0.3445
acuracia validacao: 0.3449
              precision    recall  f1-score   support

           0     0.3449    1.0000    0.5129       836
           1     0.0000    0.0000    0.0000       775
           2     0.0000    0.0000    0.0000       813

    accuracy                         0.3449      2424
   macro avg     0.1150    0.3333    0.1710      2424
weighted avg     0.1189    0.3449    0.1769      2424



## Submissão

In [9]:
tecnica = "baseline_classe_majoritaria"

metricas = {
    "tecnica": tecnica,
    "classe_codigo": "DummyClassifier",
    "acuracia_treino": acc_treino,
    "acuracia_validacao": acc_val,
    "acuracia_cv_media": None,
    "acuracia_cv_dp": None,
    "tempo_treino_s": tempo_treino,
    "hardware": "CPU local",
    "semente": SEED,
    "hiperparametros": {"strategy": "most_frequent"},
    "kaggle_publico": None,
    "data_submissao": None,
}
with open(os.path.join(RESDIR, "metricas", f"{tecnica}.json"), "w") as f:
    json.dump(metricas, f, indent=2)

predicoes_val = val[["id", "lang_abv", "label"]].copy()
predicoes_val["pred"] = pred_val
for i in range(3):
    predicoes_val[f"prob_{i}"] = prob_val[:, i]
predicoes_val.to_csv(os.path.join(RESDIR, "predicoes_val", f"{tecnica}.csv"), index=False)

submissao = pd.DataFrame({"id": teste["id"], "prediction": modelo.predict(teste[colunas_x])})
submissao.to_csv(os.path.join(SUBDIR, f"{tecnica}.csv"), index=False)
print(submissao.shape)
submissao["prediction"].value_counts()

(5195, 2)


prediction
0    5195
Name: count, dtype: int64